# Query Intent Analysis for Issue 45

Notebook-first analysis for GitHub issue 45:

- audit the published `conversation_goal` taxonomy on `train` and `test`
- label full 8-turn `test` sessions with structured conversation-dynamics tags
- pilot on about 10 sessions first
- scale to all 1000 `test` sessions once the pilot labels look usable

This notebook reads the locally cached Hugging Face Arrow files directly so it does not depend on writable access to `~/.cache` during execution.


In [ ]:
import json
import os
import time
from collections import Counter
from pathlib import Path

import pandas as pd
from datasets import Dataset
from IPython.display import display

import urllib.request
import urllib.error

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 200)
print("Ready!")


In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

HF_CACHE_ROOT = Path.home() / ".cache" / "huggingface" / "datasets"
CONV_CACHE_ROOT = HF_CACHE_ROOT / "talkpl-ai___talk_play_data-challenge-dataset" / "default" / "0.0.0"

MODEL_NAME = "openai/gpt-5.5"
API_BASE = os.environ.get("LITELLM_PROXY_BASE", "http://127.0.0.1:4001")
API_KEY = os.environ.get("LITELLM_PROXY_KEY", "sk-anything")

PILOT_SAMPLE_SIZE = 10
FULL_DATASET_LIMIT = 1000
PILOT_RANDOM_SEED = 45
TEMPERATURE = 0.0
MAX_TOKENS = 900
REQUEST_SLEEP_SECONDS = 0.0

OUTPUT_DIR = PROJECT_ROOT / "experiments" / "analysis" / "issue_45"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_FULL_DATASET = False

OUTPUT_DIR



In [ ]:
CATEGORY_LABELS = {
    "A": "Audio-Based Discovery",
    "B": "Lyrical Discovery",
    "C": "Visual-Musical Connections",
    "D": "Contextual & Situational",
    "E": "Interactive Refinement",
    "F": "Metadata-Rich Exploration",
    "G": "Mood & Emotion-Based",
    "H": "Artist & Discography Discovery",
    "I": "Cultural & Geographic",
    "J": "Social & Popularity Context",
    "K": "Temporal & Era Discovery",
}

SPECIFICITY_LABELS = {
    "LL": "Low-query / Low-target specificity",
    "HL": "High-query / Low-target specificity",
    "LH": "Low-query / High-target specificity",
    "HH": "High-query / High-target specificity",
}

SESSION_ARCHETYPES = [
    "direct_lookup",
    "exploratory_browsing",
    "iterative_refinement",
    "find_more_like_this",
    "recover_forgotten_item",
    "transition_or_pivot",
]

CONTEXT_DEPENDENCIES = [
    "none",
    "prior_user_turns",
    "prior_assistant_or_music_turns",
    "both",
]

RECOMMENDATION_MODES = [
    "single_specific_track",
    "artist_or_work_cluster",
    "multiple_valid_tracks",
]

CONSTRAINT_FACETS = [
    "mood_or_emotion",
    "lyrics_or_story",
    "era_or_time",
    "popularity_or_social",
    "geography_or_culture",
    "artist_or_discography",
    "activity_or_context",
    "multimodal_cue",
    "mixed",
]

RETRIEVAL_EVIDENCE_TYPES = [
    "exact_entity",
    "metadata_filter",
    "semantic_similarity",
    "recommendation_history_carryover",
    "hybrid",
]

FAILURE_RISKS = [
    "underspecified",
    "conflicting_constraints",
    "hidden_target",
    "long_range_callback",
    "multimodal_dependence",
]

print("Taxonomy metadata loaded.")


In [ ]:
def _find_single_arrow_file(cache_root: Path, split: str) -> Path:
    matches = sorted(cache_root.glob(f"*/talk_play_data-challenge-dataset-{split}.arrow"))
    if not matches:
        raise FileNotFoundError(f"Could not find cached Arrow file for split={split!r} under {cache_root}")
    return matches[-1]


def load_conversation_split(split: str) -> Dataset:
    arrow_path = _find_single_arrow_file(CONV_CACHE_ROOT, split)
    return Dataset.from_file(str(arrow_path))


conv_train = load_conversation_split("train")
conv_test = load_conversation_split("test")

print(f"Train sessions: {len(conv_train):,}")
print(f"Test sessions: {len(conv_test):,}")


## Goal taxonomy audit

The first pass is deterministic: compare the published goal taxonomy across `train` and `test` before any LLM labeling.


In [ ]:
def goal_taxonomy_table(dataset: Dataset, split_name: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    rows = []
    for session in dataset:
        goal = session.get("conversation_goal", {}) or {}
        rows.append(
            {
                "split": split_name,
                "category": goal.get("category"),
                "specificity": goal.get("specificity"),
            }
        )

    df = pd.DataFrame(rows)

    category = (
        df.groupby("category")
        .size()
        .rename("count")
        .reset_index()
        .sort_values(["count", "category"], ascending=[False, True])
    )
    category["pct"] = (category["count"] / len(df)).round(4)
    category["category_label"] = category["category"].map(CATEGORY_LABELS)

    specificity = (
        df.groupby("specificity")
        .size()
        .rename("count")
        .reset_index()
        .sort_values(["count", "specificity"], ascending=[False, True])
    )
    specificity["pct"] = (specificity["count"] / len(df)).round(4)
    specificity["specificity_label"] = specificity["specificity"].map(SPECIFICITY_LABELS)

    cross_tab = pd.crosstab(df["category"], df["specificity"])
    cross_tab = cross_tab.reindex(index=sorted(CATEGORY_LABELS), columns=sorted(SPECIFICITY_LABELS), fill_value=0)
    cross_tab.index.name = "category"
    cross_tab.columns.name = "specificity"

    return category, specificity, cross_tab


train_category, train_specificity, train_cross = goal_taxonomy_table(conv_train, "train")
test_category, test_specificity, test_cross = goal_taxonomy_table(conv_test, "test")

display(train_category)
display(test_category)


In [ ]:
display(train_specificity)
display(test_specificity)

train_cross


In [ ]:
test_cross


In [ ]:
def normalized_cross_tab(cross_tab: pd.DataFrame) -> pd.DataFrame:
    total = cross_tab.to_numpy().sum()
    return (cross_tab / total).round(4)


split_comparison = (
    train_category[["category", "count", "pct"]]
    .rename(columns={"count": "train_count", "pct": "train_pct"})
    .merge(
        test_category[["category", "count", "pct"]].rename(columns={"count": "test_count", "pct": "test_pct"}),
        on="category",
        how="outer",
    )
    .fillna(0)
)
split_comparison["category_label"] = split_comparison["category"].map(CATEGORY_LABELS)
split_comparison["pct_delta"] = (split_comparison["test_pct"] - split_comparison["train_pct"]).round(4)
split_comparison = split_comparison.sort_values("category")

split_comparison


## Conversation formatting and sampling

Issue 45 is explicitly about full-conversation analysis, so the LLM sees complete 8-turn sessions rather than isolated turn snapshots.


In [ ]:
def session_to_transcript(session: dict) -> str:
    lines = []
    for turn in session["conversations"]:
        role = turn["role"]
        content = turn["content"]
        lines.append(f"turn {turn['turn_number']} | {role}: {content}")
    return "\n".join(lines)


def session_overview_row(session: dict) -> dict:
    goal = session.get("conversation_goal", {}) or {}
    return {
        "session_id": session["session_id"],
        "user_id": session["user_id"],
        "category": goal.get("category"),
        "category_label": CATEGORY_LABELS.get(goal.get("category")),
        "specificity": goal.get("specificity"),
        "specificity_label": SPECIFICITY_LABELS.get(goal.get("specificity")),
        "listener_goal": goal.get("listener_goal"),
        "transcript": session_to_transcript(session),
    }


test_rows = [session_overview_row(session) for session in conv_test]
test_df = pd.DataFrame(test_rows)
test_df.head(3)


In [ ]:
pilot_df = test_df.sample(n=PILOT_SAMPLE_SIZE, random_state=PILOT_RANDOM_SEED).sort_values("session_id").reset_index(drop=True)
pilot_df[["session_id", "category", "specificity", "listener_goal"]]


In [ ]:
pilot_df.iloc[0]["transcript"][:4000]


## LiteLLM prompt and labeling helpers

The prompt requests strict JSON for the session-level labels named in issue 45.


In [ ]:
SYSTEM_PROMPT = f"""You are analyzing full conversations from a conversational music recommendation dataset.

Return exactly one JSON object with these keys:
- session_archetype: one of {SESSION_ARCHETYPES}
- context_dependency: one of {CONTEXT_DEPENDENCIES}
- recommendation_mode: one of {RECOMMENDATION_MODES}
- constraint_profile: list of one or more values from {CONSTRAINT_FACETS}
- retrieval_evidence_needed: list of one or more values from {RETRIEVAL_EVIDENCE_TYPES}
- failure_risk: list of zero or more values from {FAILURE_RISKS}
- summary: 1-2 sentence explanation

Rules:
- Base labels on the full conversation, not only the first user turn.
- Prefer the narrowest valid label set.
- Use only the allowed label values.
- Return valid JSON and nothing else.
"""


def extract_json_object(text: str) -> dict:
    text = text.strip()
    if text.startswith("```"):
        text = text.strip("`")
        if text.lower().startswith("json"):
            text = text[4:].strip()
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end < start:
        raise ValueError(f"Could not locate JSON object in response: {text[:300]}")
    return json.loads(text[start : end + 1])


def build_user_prompt(row: pd.Series) -> str:
    return f"""Session metadata
session_id: {row['session_id']}
goal_category: {row['category']} ({row['category_label']})
goal_specificity: {row['specificity']} ({row['specificity_label']})
listener_goal: {row['listener_goal']}

Conversation transcript
{row['transcript']}
"""


def call_responses_api(model_name: str, user_prompt: str) -> str:
    payload = {
        "model": model_name,
        "input": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        "max_output_tokens": MAX_TOKENS,
        "text": {"format": {"type": "json_object"}},
        "reasoning": {"effort": "low"},
    }
    request = urllib.request.Request(
        f"{API_BASE.rstrip('/')}/v1/responses",
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {API_KEY}",
        },
        method="POST",
    )
    with urllib.request.urlopen(request, timeout=180) as response:
        body = json.loads(response.read().decode("utf-8"))

    for item in body.get("output", []):
        if item.get("type") == "message":
            texts = [part.get("text", "") for part in item.get("content", []) if part.get("type") == "output_text"]
            if texts:
                return "
".join(texts)

    output_text = body.get("output_text")
    if output_text:
        return output_text

    raise ValueError(f"No output text found in responses payload: {body}")


def label_session_row(row: pd.Series, model_name: str = MODEL_NAME) -> dict:
    content = call_responses_api(model_name=model_name, user_prompt=build_user_prompt(row))
    parsed = extract_json_object(content)
    parsed["session_id"] = row["session_id"]
    parsed["goal_category"] = row["category"]
    parsed["goal_specificity"] = row["specificity"]
    parsed["model_name"] = model_name
    return parsed




In [ ]:
def sanitize_name(value: str) -> str:
    return value.replace("/", "_").replace(":", "_").replace("-", "_")


def load_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    rows = []
    with path.open() as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def append_jsonl(path: Path, row: dict) -> None:
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")


def run_labeling(df: pd.DataFrame, output_path: Path, model_name: str = MODEL_NAME) -> pd.DataFrame:
    existing_rows = load_jsonl(output_path)
    completed_ids = {row["session_id"] for row in existing_rows}
    rows = list(existing_rows)

    for idx, row in enumerate(df.itertuples(index=False), start=1):
        if row.session_id in completed_ids:
            continue
        labeled = label_session_row(pd.Series(row._asdict()), model_name=model_name)
        append_jsonl(output_path, labeled)
        rows.append(labeled)
        completed_ids.add(row.session_id)
        print(f"[{idx}/{len(df)}] labeled {row.session_id}")
        if REQUEST_SLEEP_SECONDS:
            time.sleep(REQUEST_SLEEP_SECONDS)

    return pd.DataFrame(rows)


pilot_output_path = OUTPUT_DIR / f"pilot_labels_{sanitize_name(MODEL_NAME)}_{PILOT_SAMPLE_SIZE}.jsonl"
pilot_output_path


## Pilot run on about 10 sessions

Run this first. Review the outputs before switching to the full 1000-session pass.


In [ ]:
pilot_labels_df = run_labeling(pilot_df, pilot_output_path, model_name=MODEL_NAME)
pilot_labels_df


In [ ]:
for column in ["session_archetype", "context_dependency", "recommendation_mode"]:
    print(f"\n{column}")
    display(pilot_labels_df[column].value_counts(dropna=False).rename_axis(column).reset_index(name="count"))


In [ ]:
pilot_review = pilot_df.merge(pilot_labels_df, on="session_id", how="left")
pilot_review[
    [
        "session_id",
        "category",
        "specificity",
        "session_archetype",
        "context_dependency",
        "recommendation_mode",
        "constraint_profile",
        "retrieval_evidence_needed",
        "failure_risk",
        "summary",
    ]
]


## Full 1000-session labeling pass

After the pilot looks good, set `RUN_FULL_DATASET = True` in the config cell and execute the next cells.


In [ ]:
full_df = test_df.sort_values("session_id").head(FULL_DATASET_LIMIT).reset_index(drop=True)
full_output_path = OUTPUT_DIR / f"full_labels_{sanitize_name(MODEL_NAME)}_{FULL_DATASET_LIMIT}.jsonl"

if RUN_FULL_DATASET:
    full_labels_df = run_labeling(full_df, full_output_path, model_name=MODEL_NAME)
else:
    full_labels_df = pd.DataFrame(load_jsonl(full_output_path))
    print(
        "RUN_FULL_DATASET is False. Loaded existing rows:" if not full_labels_df.empty else "RUN_FULL_DATASET is False. No full-run file yet.",
        len(full_labels_df),
    )

full_labels_df.head()


In [ ]:
if not full_labels_df.empty:
    full_labeled = full_df.merge(full_labels_df, on="session_id", how="left")
    display(full_labeled[["session_id", "category", "specificity", "session_archetype", "context_dependency", "recommendation_mode"]].head())

    for column in ["session_archetype", "context_dependency", "recommendation_mode"]:
        print(f"\n{column}")
        display(full_labeled[column].value_counts(dropna=False).rename_axis(column).reset_index(name="count"))
else:
    print("Run the full labeling cell first.")


In [ ]:
if not full_labels_df.empty:
    category_by_archetype = pd.crosstab(full_labeled["category"], full_labeled["session_archetype"]).sort_index()
    specificity_by_mode = pd.crosstab(full_labeled["specificity"], full_labeled["recommendation_mode"]).sort_index()
    display(category_by_archetype)
    display(specificity_by_mode)
else:
    print("Run the full labeling cell first.")


In [ ]:
def flatten_counter(series: pd.Series) -> pd.DataFrame:
    counter = Counter()
    for value in series.dropna():
        if isinstance(value, list):
            counter.update(value)
    return (
        pd.DataFrame(sorted(counter.items(), key=lambda item: (-item[1], item[0])), columns=["label", "count"])
        if counter
        else pd.DataFrame(columns=["label", "count"])
    )


if not full_labels_df.empty:
    display(flatten_counter(full_labeled["constraint_profile"]))
    display(flatten_counter(full_labeled["retrieval_evidence_needed"]))
    display(flatten_counter(full_labeled["failure_risk"]))
else:
    print("Run the full labeling cell first.")


## Optional markdown export

This cell writes a compact summary markdown artifact after the full run is available.


In [ ]:
def write_summary_markdown(path: Path, full_labeled: pd.DataFrame) -> Path:
    archetype_counts = full_labeled["session_archetype"].value_counts(dropna=False)
    context_counts = full_labeled["context_dependency"].value_counts(dropna=False)
    mode_counts = full_labeled["recommendation_mode"].value_counts(dropna=False)

    lines = [
        "# Query Intent Analysis v1",
        "",
        f"- model: `{MODEL_NAME}`",
        f"- sessions labeled: `{len(full_labeled)}`",
        "",
        "## Session archetypes",
    ]
    lines.extend([f"- {label}: {count}" for label, count in archetype_counts.items()])
    lines.append("")
    lines.append("## Context dependency")
    lines.extend([f"- {label}: {count}" for label, count in context_counts.items()])
    lines.append("")
    lines.append("## Recommendation mode")
    lines.extend([f"- {label}: {count}" for label, count in mode_counts.items()])

    path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return path


summary_path = OUTPUT_DIR / "query_intent_v1.md"
if not full_labels_df.empty:
    write_summary_markdown(summary_path, full_labeled)
summary_path
